## **Projeto:** Merca Data Platform

##**Squad:** 2 | Camada Bronze
### Objetivo
Ingerir os dados brutos do catálogo de produtos na camada Bronze do Delta Lake.
Nenhuma transformação é aplicada ao conteúdo — apenas colunas de auditoria e particionamento temporal são adicionados para otimizar consultas futuras.
### Origem e Destino
| **Origem** | `real-time-data/<snapshot>/ecommerce_produtos.parquet` (ADLS) |
| **Destino** | `squad2/bronze/ecommerce_produtos` (Delta Lake) |
| **Checkpoint** | `squad2/control/ecommerce_produtos/control_file.json` |
| **Modo de escrita** | `append` incremental por snapshot |
| **Particionamento** | `ingestion_year / ingestion_month / ingestion_day / ingestion_hour` |
### Colunas de Auditoria Adicionadas
| Coluna | Descrição |
| `bronze_source_file` | Caminho completo do arquivo Parquet de origem |
| `bronze_ingested_at` | Timestamp de ingestão na Bronze |
| `_source` | Fonte dos dados (`real-time-data`) |
| `_camada` | Camada atual (`bronze`) |
| `ingestion_year` | Ano de ingestão (usado como partição) |
| `ingestion_month` | Mês de ingestão (usado como partição) |
| `ingestion_day` | Dia de ingestão (usado como partição) |
| `ingestion_hour` | Hora de ingestão (usado como partição) |
### Dependências
| Notebook | Motivo |
| `feat_squad2_99_helpers` | Conexão ADLS, leitura Parquet, escrita Delta, checkpoint e log |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import logging
from pyspark.sql.functions import lit, current_timestamp

logging.getLogger("azure").setLevel(logging.WARNING)

TABELA      = "ecommerce_produtos"
BRONZE_PATH = get_delta_path("bronze", TABELA)

inicio = log_inicio(f"feat_squad2_bronze_{TABELA}")

log.info(f"Tabela      : {TABELA}")
log.info(f"Bronze Path : {BRONZE_PATH}")

In [0]:
try:
    snapshots = sorted(listar_snapshots())
    log.info(f"{len(snapshots)} snapshot(s) disponível(is):\n")
    for snap in snapshots:
        print(f"  Pacote {snap}")

except Exception as e:
    log.error(f"Erro ao listar snapshots: {str(e)}")
    raise

In [0]:
try:
    # Esta função lê automaticamente de control/ecommerce_produtos/control_file.json
    processados = ler_checkpoint("bronze", TABELA)
    novos       = [s for s in snapshots if s not in processados]
    log.info(f"{len(novos)} snapshot(s) novo(s) para processar.")

except Exception as e:
    log.error(f"Erro ao verificar controle: {str(e)}")
    raise

In [0]:
try:
    total_linhas = 0

    if not novos:
        log.info(" Sistema em dia! Nenhum snapshot novo para processar no momento.")
    else:
        for snapshot_id in novos:
            log.info(f"Processando snapshot: {snapshot_id}")

            # 1. Lê o arquivo bruto via Spark (Mantém o processamento distribuído)
            df = ler_parquet(snapshot_id, TABELA)

            # 2. Adiciona colunas de auditoria e chaves de partição de tempo (Sem transformações)
            from pyspark.sql.functions import (
                lit, current_timestamp,
                year, month, dayofmonth, hour
            )

            df_bronze = df \
                .withColumn(
                    "bronze_source_file",
                    lit(f"{PATHS['raw']}/{snapshot_id}/{TABELA}.parquet")
                ) \
                .withColumn("bronze_ingested_at", current_timestamp()) \
                .withColumn("_source",            lit("real-time-data")) \
                .withColumn("_camada",            lit("bronze")) \
                .withColumn("ingestion_year",     year(current_timestamp()).cast("string")) \
                .withColumn("ingestion_month",    month(current_timestamp()).cast("string")) \
                .withColumn("ingestion_day",      dayofmonth(current_timestamp()).cast("string")) \
                .withColumn("ingestion_hour",     hour(current_timestamp()).cast("string"))

            # 3. Grava como Delta de forma incremental (Modo append inteligente e estável)
            sucesso       = gravar_delta(df_bronze, "bronze", TABELA, mode="append")
            count         = df_bronze.count()
            total_linhas += count
            processados.add(snapshot_id)

            log.info(f"   OK {snapshot_id} → {count} linhas integradas.")

        # 4. Atualiza a pasta control/ com o histórico rico em JSON
        salvar_checkpoint("bronze", TABELA, processados)
        log.info(f" Ingestão Concluída! Total gravado nesta rodada: {total_linhas} linhas.")

except Exception as e:
    log.error(f"Erro na ingestão Bronze da tabela {TABELA}: {str(e)}")
    raise

In [0]:
try:
    # Leitura robusta com consolidação automática de chunks de partições
    df_bronze = ler_delta("bronze", TABELA)
    total     = df_bronze.count()

    log.info(f"   Validação Bronze OK!")
    log.info(f"   Path            : {BRONZE_PATH}")
    log.info(f"   Total registros : {total}")
    log.info(f"   Colunas         : {len(df_bronze.columns)}")

    print("\n Schema da Tabela na Bronze:")
    df_bronze.printSchema()

    print("\n Amostra dos Dados Gravados:")
    display(df_bronze)

except Exception as e:
    log.error(f"Erro na validação da tabela: {str(e)}")
    raise

In [0]:
try:
    squad2_client = get_squad2_client()
    paths         = list(squad2_client.get_paths(
        path      = f"bronze/{TABELA}",
        recursive = True
    ))

    parquets = [p for p in paths if p.name.endswith(".parquet")]

    log.info(f" Histórico de arquivos físicos gravados no Lake: {len(parquets)} arquivo(s)\n")
    for p in sorted(parquets, key=lambda x: x.name):
        tamanho = p.content_length if p.content_length else 0
        print(f"  Arquivo: {p.name.split('bronze/')[1]} | {tamanho} bytes")

except Exception as e:
    log.warning(f" Histórico ignorado: {str(e)[:80]}")

In [0]:
log_fim(f"feat_squad2_bronze_{TABELA}", inicio)